# Objetivo

Desenvolvimento do Teleutils em ambiente de teste local

# Configuração do Ambiente

## Bibliotecas

In [ ]:
# 1. Módulos da Biblioteca Padrão
import logging
import json
from pathlib import Path


# 2. Módulos de Terceiros
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf
from pyspark.sql import types as T # ArrayType, StructType, StructField, StringType, BooleanType

# 3. Teletools/Teleutils
from teleutils.core.extractors import CDRParquetExtractor, CDRTextExtractor
from teleutils.core.transformers import CDRTransformer

In [ ]:
# Configuração do Pandas
# Exibir todas as colunas
pd.set_option("display.max_columns", None)

# Exibir todas as linhas
pd.set_option("display.max_rows", None)

# Evitar que o texto nas células seja cortado
pd.set_option("display.max_colwidth", None)

In [ ]:
# Configuração para exibir logs no output da célula
logging.basicConfig(
 level=logging.INFO,
 format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
 datefmt="%H:%M:%S",
)

# Reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
# logging.getLogger("py4j").setLevel(logging.WARNING)
# logging.getLogger("pyspark").setLevel(logging.WARNING)

## Spark

In [ ]:
spark = (
 SparkSession
 .builder
 .appName("SessaoLocal")
 .master("local[12]")
 .config("spark.driver.memory", "24g")
 .getOrCreate()
)
spark

# Análise

## Configuração

In [ ]:
file_config = "/workspaces/teleutils/src/teleutils/core/orchestrators/etl.config.example.json"
df_config = pd.read_json(file_config)
df_config

In [ ]:
with open(file_config) as f:
    config = json.load(f)
[item for item in config if item["output_file_prefix"]=="algar_stfc_huawei_ngn"][0]

In [ ]:
config_algar_stfc_local = {
    "source_folder": "/assets/monitoring/cdr/cdr_bruto/Semana100/Algar/FIXA/NGN",
    "pattern": "*.gz",
    "schema": "stfc_huawei_ngn",
    "extracted_folder": "/assets/data/temp/cdr_stfc/extracted",
    "transformed_folder": "/assets/data/temp/cdr_stfc/transformed",
    "output_file_prefix": "algar_stfc_huawei_ngn",
    "storage_type": "local"
}

In [ ]:
from teleutils.core.orchestrators import get_config
get_config(config_algar_stfc_local)

## Extração e transformação

### Um item

In [ ]:
from teleutils.core.orchestrators import run_etl_item

In [ ]:
run_etl_item(config_algar_stfc_local, spark=spark)

In [ ]:
extracted = "file:///assets/data/temp/cdr_stfc/extracted/algar_stfc_huawei_ngn_extracted.parquet"
transformed = "file:///assets/data/temp/cdr_stfc/transformed/algar_stfc_huawei_ngn_transformed.parquet"

In [ ]:
df_extracted = spark.read.parquet(extracted)
df_extracted.show(5)

In [ ]:
df_transformed = spark.read.parquet(transformed)
df_transformed.limit(5).toPandas()

### Vários itens

In [ ]:
configs = [
    {
        "source_folder": "/assets/monitoring/cdr/cdr_bruto/Semana100/Tim/FIXA/NGN_Huawei",
        "pattern": "*.gz",
        "schema": "stfc_huawei_ngn_tim",
        "extracted_folder": "/assets/data/temp/cdr_stfc/extracted",
        "transformed_folder": "/assets/data/temp/cdr_stfc/transformed",
        "output_file_prefix": "stfc_huawei_ngn_tim",
        "storage_type": "local"
    },
    {
        "source_folder": "/assets/monitoring/cdr/cdr_bruto/Semana100/Oi/CDR",
        "pattern": "*TN*.gz",
        "schema": "stfc_tropico_oi",
        "extracted_folder": "/assets/data/temp/cdr_stfc/extracted",
        "transformed_folder": "/assets/data/temp/cdr_stfc/transformed",
        "output_file_prefix": "stfc_tropico_oi",
        "storage_type": "local"
    },
]

In [ ]:
from teleutils.core.orchestrators import run_etl

In [ ]:
results = run_etl(configs=configs, spark=spark)
results

In [ ]:
for item in results.items:
    print(f"\nItem {item.index} — schema: {item.schema!r}")
    print(f"  Status: {item.status}")
    print(f"  Etapa: {item.stage}")
    print(f"  Etapas concluídas: {item.completed_stages}")

    if item.paths is not None:
        print(f"  Origem: {item.paths.source_file}")
        print(f"  Extraído: {item.paths.extracted_file}")
        print(f"  Transformado: {item.paths.transformed_file}")

    if item.error_type is not None:
        print(f"  Erro: {item.error_type}: {item.error_message}")

print(
    f"\nResumo: total={results.total}, sucessos={results.successes}, "
    f"erros de validação={results.validation_errors}, "
    f"erros de execução={results.execution_errors}"
)

In [ ]:
import logging

from teleutils.core.orchestrators import ETLReport

logger = logging.getLogger(__name__)


def print_etl_report(
    results: ETLReport,
    *,
    print_to_log: bool = False,
) -> None:
    linhas = []

    for item in results.items:
        linhas.extend(
            [
                f"Item {item.index} — schema: {item.schema!r}",
                f"  Status: {item.status}",
                f"  Etapa: {item.stage}",
                f"  Etapas concluídas: {item.completed_stages}",
            ]
        )

        if item.paths is not None:
            linhas.extend(
                [
                    f"  Origem: {item.paths.source_file}",
                    f"  Extraído: {item.paths.extracted_file}",
                    f"  Transformado: {item.paths.transformed_file}",
                ]
            )

        if item.error_type is not None:
            linhas.append(f"  Erro: {item.error_type}: {item.error_message}")

        linhas.append("")

    linhas.append(
        f"Resumo: total={results.total}, sucessos={results.successes}, "
        f"erros de validação={results.validation_errors}, "
        f"erros de execução={results.execution_errors}"
    )

    for linha in linhas:
        if print_to_log:
            logger.info("%s", linha)
        else:
            print(linha)

In [ ]:
print_etl_report(results)

In [ ]:
print_etl_report(results, print_to_log=True)